# Sesión 01 en vivo — LLMs y Setup de Entorno de Trabajo

**Programa AI Engineering — LIDR**

Este notebook contiene todos los ejercicios de la sesión en vivo. Ejecuta las celdas de configuración primero y luego ve avanzando bloque por bloque con el instructor.

---

> ## 🔧 Versión corregida
>
> Es una copia del notebook de la sesión en vivo del tutor (`Copia_de_LIDR___AI_Engineering_S1_sesion_01_notebook_sesion_en_vivo.ipynb`, que se conserva sin cambios como referencia). El contenido es el mismo y solo cambian estos puntos, marcados con **🔧 Corrección N** donde aparecen:
>
> | # | Problema en el original | Corrección |
> |---|---|---|
> | 1 | Solo funcionaba en Colab: importaba `google.colab` sin alternativa | Usa Colab Secrets en Colab y `.env` + `uv sync` en local |
> | 2 | `PRICING` definido dos veces y con un id inexistente (`claude-sonnet-4-6-20250514`) | Una sola tabla con ids base |
> | 3 | `calculate_cost` buscaba por subcadena en el orden del dict, así que `gpt-5.4-nano` cobraba el precio de `gpt-5.4` y `gpt-4o` el de `gpt-4o-mini` | `find_price` busca el prefijo más largo |
> | 4 | `response.content[0].text` falla si el primer bloque no es texto (modelos con *thinking*, herramientas) | Helper `anthropic_text()` que filtra los bloques `type == "text"` |
> | 5 | Las llamadas a Anthropic con `max_tokens=400` no comprobaban `stop_reason` | `warn_if_truncated()` avisa si la respuesta se cortó |
> | 6 | La sección de `temperature` no menciona que los Claude recientes la rechazan | Nota en la sección 3.1 |
> | 7 | Comentarios "Uncomment if you have access" en líneas que ya estaban activas | Comentarios aclarados |
> | 8 | f-string con comillas anidadas en la comparación lado a lado: solo es válido en Python 3.12+ y da `SyntaxError` en 3.11 | Formateo en variables previas |
>
> **Ejecutar en local:** `cd session_1 && uv sync`, con las keys en `.env`.

## Configuración inicial

Ejecuta estas tres celdas antes de empezar. Necesitas tener tus API keys configuradas en **Colab Secrets** (icono de llave 🔑 en el panel lateral):
- `OPENAI_API_KEY` → tu key de OpenAI (empieza por `sk-`)
- `ANTHROPIC_API_KEY` → tu key de Anthropic (empieza por `sk-ant-`)

In [1]:
# Install dependencies
# 🔧 Corrección 1: en local las dependencias las instala `uv sync` (pyproject.toml).
# Solo instalamos con pip cuando estamos en Colab.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Dependency warnings from Colab pre-installed packages (gradio, typer) are normal and do not affect our code
    get_ipython().system("pip install -q openai anthropic tiktoken litellm 2>&1 | grep -v 'dependency' | grep -v 'WARNING' | tail -1")

# Verify installation
import openai, anthropic, tiktoken, litellm
from importlib.metadata import version
print(f"Entorno: {'Google Colab' if IN_COLAB else 'local'}")
print(f"✅ openai {openai.__version__}")
print(f"✅ anthropic {anthropic.__version__}")
print(f"✅ tiktoken {version('tiktoken')}")
print(f"✅ litellm {version('litellm')}")

Entorno: local
✅ openai 2.54.0
✅ anthropic 2.4.0
✅ tiktoken 0.14.0
✅ litellm 1.106.6


In [2]:
# Load API keys
# 🔧 Corrección 1: el original importaba `google.colab` sin alternativa, así que fallaba
# fuera de Colab. Ahora lee Colab Secrets en Colab y el archivo `.env` en local.
import os

if IN_COLAB:
    from google.colab import userdata

    for key in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        os.environ[key] = userdata.get(key)
else:
    from dotenv import load_dotenv

    load_dotenv()  # lee session_1/.env

missing = [k for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY") if not os.getenv(k)]
assert not missing, f"Faltan API keys: {missing}"

# Create clients
from openai import OpenAI
from anthropic import Anthropic

openai_client = OpenAI()
anthropic_client = Anthropic()

print("✅ Clients configured successfully")

✅ Clients configured successfully


In [3]:
# Utility functions used throughout the session

# 🔧 Corrección 2: una única tabla de precios. El original la redefinía en el bloque 4.
# Las claves son ids base de modelo, sin fecha. La API puede devolver el id con fecha
# (p. ej. gpt-4o-mini-2024-07-18) y lo resolvemos por prefijo en `find_price`.
# También quitamos "claude-sonnet-4-6-20250514", un id que no existe.
# USD por 1M tokens. ⚠️ Verifica los precios en las páginas oficiales: cambian.
PRICING = {
    # OpenAI
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "gpt-5.4-nano": {"input": 0.20, "output": 1.25},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "output": 0.40},
    # Anthropic
    "claude-haiku-4-5": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
}


def find_price(model):
    """🔧 Corrección 3: busca el precio por el prefijo conocido más largo.

    El original buscaba por subcadena (`key in model or model in key`) en el orden del
    dict. Así, "gpt-5.4-nano-<fecha>" coincidía antes con "gpt-5.4" (12x más caro) y
    "gpt-4o" coincidía con "gpt-4o-mini" (precio de otro modelo). Con el prefijo más
    largo, cada id se resuelve a su propio modelo, o a None si no está en la tabla.
    """
    if model in PRICING:
        return PRICING[model]
    matches = [key for key in PRICING if model.startswith(key)]
    return PRICING[max(matches, key=len)] if matches else None


def calculate_cost(model, input_tokens, output_tokens):
    """Calculate the cost of an API call in USD."""
    prices = find_price(model)
    if not prices:
        return None

    cost_in = (input_tokens / 1_000_000) * prices["input"]
    cost_out = (output_tokens / 1_000_000) * prices["output"]
    return {"input": cost_in, "output": cost_out, "total": cost_in + cost_out}


def anthropic_text(response):
    """🔧 Corrección 4: une los bloques de texto de una respuesta de Anthropic.

    El original usaba `response.content[0].text`, que asume que el primer bloque es
    texto. Con modelos que razonan (Claude Haiku 5.5, Opus 5.5...) el primer bloque es
    `thinking`, y eso daba AttributeError. Lo mismo pasa con respuestas que usan
    herramientas.
    """
    return "".join(block.text for block in response.content if block.type == "text")


def warn_if_truncated(response):
    """🔧 Corrección 5: avisa si Anthropic cortó la respuesta por `max_tokens`."""
    if response.stop_reason == "max_tokens":
        print("⚠️  Respuesta truncada: se alcanzó max_tokens (stop_reason='max_tokens')")


print("✅ Cost calculation function loaded")

✅ Cost calculation function loaded


---
# Bloque 2 — Del prompt del usuario al prompt del producto

**Objetivo:** La calidad de la respuesta no puede depender de que el usuario sepa hacer prompting. El desarrollador es quien diseña la experiencia.

### 2.1. El problema: la respuesta genérica

Enviamos una pregunta realista pero sin system prompt. ¿Pondríais esta respuesta en un producto?

In [4]:
# No system prompt — the model has no context about what we need

MODEL = "gpt-4o-mini"

response = openai_client.responses.create(
    model=MODEL,
    input="We're migrating our payment service from a monolith to microservices. What should we consider?"
)

print(response.output_text)
print(f"\n--- {response.usage.input_tokens} input tokens + {response.usage.output_tokens} output tokens ---")
print(f"\n--- [Cost: {calculate_cost(MODEL, response.usage.input_tokens, response.usage.output_tokens)}]")

Migrating a payment service from a monolithic architecture to microservices is a complex but rewarding journey. Here are key considerations to help guide your migration:

### 1. **Service Boundaries**
   - **Define Microservices**: Identify distinct business capabilities (like payment processing, fraud detection, user accounts, etc.) that can be encapsulated as independent services.
   - **Decompose Gradually**: Consider starting with less critical services to minimize risks and understand the migration's implications.

### 2. **Data Management**
   - **Database Strategies**: Decide whether to adopt a single database for all services, a separate database per service, or a combination (e.g., shared schema for common data).
   - **Data Consistency**: Implement strategies for managing data consistency across services, such as eventual consistency or using event sourcing.

### 3. **Communication Patterns**
   - **Synchronous vs. Asynchronous**: Determine if services will communicate synchr

### 2.2. El system prompt como contrato de calidad

Vamos a enviar la **misma pregunta** con system prompts cada vez más elaborados. Cada iteración añade una capa de control sobre la respuesta.

In [5]:
# The question stays the same across all iterations
USER_QUESTION = "We're migrating our payment service from a monolith to microservices. What should we consider?"

In [6]:
# Iteration 1 — Basic role: slightly better, but still unfocused
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a senior software architect with experience in distributed systems.",
    input=USER_QUESTION
)

print("=== ITERATION 1: Basic role ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 1: Basic role ===
Migrating from a monolithic payment service to a microservices architecture is a significant undertaking that requires careful planning and execution. Here are key considerations to guide you through this transition:

### 1. **Service Decomposition**
   - **Identify Bounded Contexts**: Analyze your monolith to identify distinct business capabilities that can be transformed into microservices (e.g., payment processing, transaction history, user management).
   - **Define APIs**: Clearly define the APIs for each microservice, ensuring that they are cohesive and focused on specific tasks.

### 2. **Data Management**
   - **Database Strategy**: Decide on a data management strategy. Each microservice should ideally have its own database to ensure decoupling, which may involve migrating from a single database to multiple databases.
   - **Data Consistency**: Implement strategies like eventual consistency and distributed transactions management using patterns l

In [7]:
# Iteration 2 — Role + audience + format constraints
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="""You are a senior software architect with 15+ years of experience in distributed systems and payment infrastructure.

Your audience is a team of experienced backend developers (5+ years) who understand software fundamentals but have never done a monolith-to-microservices migration.

Response format:
- Write in prose, no bullet points or numbered lists
- Maximum 250 words
- Prioritize actionable insights over generic theory""",
    input=USER_QUESTION
)

print("=== ITERATION 2: Role + audience + format ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 2: Role + audience + format ===
When migrating your payment service from a monolith to microservices, focus on a gradual and strategic approach to minimize risk and ensure stability. Start by identifying the core functionalities within your payment system, such as transaction processing, fraud detection, and user management. Break these down into well-defined microservices, ensuring each one has a single responsibility and can operate autonomously.

Before diving into the migration, establish clear communication between the teams handling different services. This includes defining APIs and data contracts early on, as they will serve as the backbone of interaction between microservices. Consider implementing an API gateway to manage requests, which can also help with cross-cutting concerns like authentication, rate limiting, and logging.

Next, pay special attention to your data management strategy. Many monolithic systems use a single database, which can lead to coupling.

In [8]:
# Iteration 3 — Adding behavioral constraints and edge case handling
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="""You are a senior software architect with 15+ years of experience in distributed systems and payment infrastructure. You have led multiple monolith-to-microservices migrations at fintech companies.

Your audience is a team of experienced backend developers (5+ years) who understand software fundamentals but have never done a monolith-to-microservices migration.

Response rules:
- Write in prose, no bullet points or numbered lists
- Maximum 250 words
- Prioritize actionable insights over generic theory
- When discussing trade-offs, always present both sides with concrete consequences
- If the question lacks critical context (team size, traffic volume, regulatory requirements), explicitly state what assumptions you are making
- Never recommend a specific vendor or product — focus on architectural patterns
- Always mention at least one non-obvious risk that teams commonly overlook""",
    input=USER_QUESTION
)

print("=== ITERATION 3: Role + audience + format + constraints ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 3: Role + audience + format + constraints ===
Migrating your payment service from a monolith to microservices requires careful consideration of several factors. I presume you're part of a medium-sized team, dealing with moderate traffic, and are subject to stringent regulatory requirements. One key aspect to address is data management. In a monolith, you likely have a unified database, but in microservices, each service should ideally have its own database. This can lead to eventual consistency issues, as transactions might span multiple services. You'll need a robust strategy for managing these transactions, perhaps leaning towards sagas or 2PC, but be prepared for the complexity they add.

Consider service boundaries carefully. Too granular a decomposition can lead to microservices sprawl, making the system harder to manage and increasing inter-service communication overhead, which then impacts latency. Conversely, larger services could result in the return of monolithi

In [9]:
# Iteration 4 — Production-grade prompt: full quality contract
PRODUCTION_SYSTEM_PROMPT = """You are a principal software architect specializing in payment systems, PCI-DSS compliance, and large-scale distributed architectures. You have led monolith-to-microservices migrations at three fintech companies processing over $1B in annual transactions.

Audience: senior backend developers (5+ years experience) planning their first major architectural migration. They understand distributed systems theory but need practical, battle-tested guidance.

Response structure:
- Open with the single most critical decision they need to make first
- Follow with 2-3 key considerations, each grounded in a concrete scenario or real-world consequence
- Close with one commonly overlooked risk and how to mitigate it
- Write in prose, no bullet points, no numbered lists, no markdown headers
- Maximum 300 words

Behavioral constraints:
- When you lack context (team size, current traffic, regulatory environment), state your assumptions explicitly before advising
- Never recommend specific vendors, cloud providers, or commercial products
- Present trade-offs with concrete consequences: 'If you choose X, expect Y; if you choose Z, expect W'
- Distinguish between what is essential on day one versus what can be deferred
- If the migration scope is ambiguous, ask one clarifying question before providing your analysis

Do not:
- Use filler phrases like 'Great question!' or 'There are many things to consider'
- Repeat information the user already provided
- Give generic advice that applies to any software project, not specifically to payment system migrations"""

response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=PRODUCTION_SYSTEM_PROMPT,
    input=USER_QUESTION
)

print("=== ITERATION 4: Production-grade prompt ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

=== ITERATION 4: Production-grade prompt ===
The most critical decision you need to make first is whether to pursue a "big bang" migration or a gradual, incremental approach. If you opt for the former, expect significant downtime and increased risk of disrupting ongoing transactions; if you choose the latter, be prepared for longer delivery timelines but a more controlled and less risky transition.

As a key consideration, determine your domain boundaries early. For example, if you decide to split payment processing from fraud detection, you will need to manage inter-service communication carefully. Poorly defined boundaries can lead to tightly coupled services, which can negate the benefits of microservices and complicate future scaling. Expect a performance hit if data needs to be shared across services too often; you'll have to balance efficiency with the independence microservices are meant to provide.

Another consideration is your data management strategy. In a monolithic system,

### 2.3. Abstraer al usuario del prompting

En un producto real, el usuario no escribe prompts. Solo aporta datos mínimos. El desarrollador construye el prompt internamente. Veamos dos versiones de la misma funcionalidad: una amateur y una profesional.

In [10]:
# NAIVE version — minimal prompt, inconsistent results

def review_pull_request_naive(diff_text: str) -> str:
    """Naive approach: vague instructions, unpredictable output."""
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="You are a code reviewer.",
        input=f"Review this code:\n\n{diff_text}",
        temperature=0.7
    )
    return response.output_text


sample_diff = """def process_payment(amount, currency, user_id):
    if amount > 0:
        result = gateway.charge(amount, currency)
        db.save(user_id, result)
        return result
    return None"""

print("=== NAIVE VERSION ===")
print(review_pull_request_naive(sample_diff))

=== NAIVE VERSION ===


This code snippet seems to be part of a payment processing system. Here's a review of the code with some suggestions for improvement:

### Code Review

1. **Functionality**:
   - The function `process_payment` checks if the amount is greater than zero before attempting to charge the user. It seems to be handling a basic payment flow.

2. **Naming**:
   - The function name `process_payment` is clear and descriptive, indicating that it handles payment processing.

3. **Parameters**:
   - The parameters `amount`, `currency`, and `user_id` are relevant and necessary for the payment process.

4. **Error Handling**:
   - There is no error handling for the payment process. If `gateway.charge` fails (e.g., due to network issues, invalid parameters, etc.), the current implementation will not handle that gracefully. Consider wrapping the charge call in a try-except block.

5. **Validation**:
   - You might want to validate the `currency` and `user_id` parameters to ensure they are correct before

In [11]:
# PRODUCTION version — detailed prompt, consistent and actionable output

def review_pull_request(diff_text: str, context: str = "") -> str:
    """
    Production-grade PR review.
    The user pastes a diff. The developer controls everything else.
    """
    instructions = """You are a staff engineer performing a pull request review for a fintech company that processes credit card payments. Your reviews are known for being thorough but respectful.

Review criteria (in priority order):
1. Security vulnerabilities: injection, authentication gaps, sensitive data exposure
2. Error handling: missing try/catch, silent failures, unhandled edge cases
3. Data integrity: race conditions, transaction boundaries, idempotency
4. Code clarity: naming, function length, single responsibility

Response format:
- Start with a one-sentence overall assessment (approve, request changes, or needs discussion)
- For each finding: describe the issue, explain the concrete risk, suggest a fix with a brief code snippet if applicable
- Maximum 3 findings — prioritize by severity, skip nitpicks
- Write in prose, no markdown headers, no bullet points

Constraints:
- Never say 'looks good' if there are security or data integrity concerns
- If the diff is too small to identify the full context, state what assumptions you are making
- Do not suggest style changes unless they affect readability in a meaningful way
- Treat the author as a competent developer — explain the 'why', not the 'how to write code'"""

    user_input = f"Pull request diff to review:\n\n```\n{diff_text}\n```"
    if context:
        user_input += f"\n\nAdditional context: {context}"

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=instructions,
        input=user_input,
        temperature=0.15,
        max_output_tokens=600
    )
    return response.output_text


# The user only pastes a diff — the developer controls everything else
print("=== PRODUCTION VERSION ===")
print(review_pull_request(sample_diff, context="This function runs inside our payment processing pipeline."))

print("\n" + "=" * 60)
print("The user pasted 6 lines of code.")
print("The model received a 250-word system prompt + formatted input + context.")

=== PRODUCTION VERSION ===


I request changes to this pull request due to potential security and data integrity concerns.

First, the function lacks proper error handling for the payment processing. If the `gateway.charge` method fails (e.g., due to network issues or invalid payment details), the function will not handle the exception, leading to silent failures. This could result in users being charged without a proper record in the database, creating confusion and potential financial discrepancies. To mitigate this risk, you should wrap the payment processing in a try/catch block to handle exceptions gracefully and log any errors for further investigation. For example:

```python
try:
    result = gateway.charge(amount, currency)
except Exception as e:
    log.error(f"Payment processing failed for user {user_id}: {e}")
    return None
```

Second, there is a potential issue with data integrity regarding transaction boundaries. If the payment is successfully charged but the database save operation fails, you cou

### 🧪 Tu turno

Crea tu propia función que abstraiga el prompting con un prompt de nivel producción. Algunas ideas:
- Analizar un requisito de software y detectar ambigüedades
- Evaluar un diseño de base de datos y sugerir mejoras
- Revisar un mensaje de error de cara al usuario y proponer uno mejor
- Analizar un endpoint API y evaluar su diseño REST

In [12]:
# Your production-grade function here

def my_function(user_input: str) -> str:
    instructions = """..."""

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=instructions,
        input=user_input,
        temperature=0.2,
        max_output_tokens=500
    )
    return response.output_text

# Test:
# print(my_function("..."))

### 2.4. Consistencia de respuestas

Los modelos son estocásticos. ¿Es aceptable que dos usuarios reciban respuestas de calidad diferente ante la misma pregunta? Veamos cómo la combinación de temperature + instrucciones precisas controla la variabilidad.

In [13]:
# High temperature + vague prompt = wildly inconsistent responses
print("=== HIGH TEMPERATURE (1.0) + VAGUE PROMPT ===")
for i in range(4):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="You are a technical consultant.",
        input="Should we use microservices?",
        temperature=1.0
    )
    # Show first 120 chars to compare at a glance
    text = response.output_text.replace("\n", " ")[:120]
    print(f"  Run {i+1}: {text}...")

=== HIGH TEMPERATURE (1.0) + VAGUE PROMPT ===


  Run 1: Deciding whether to use microservices depends on several factors related to your specific project requirements, team cap...


  Run 2: Deciding whether to adopt a microservices architecture depends on several factors specific to your organization, project...


  Run 3: Deciding whether to use microservices depends on several factors related to your organization's goals, architecture, and...


  Run 4: Deciding whether to use microservices depends on several factors specific to your project and organizational needs. Here...


In [14]:
# Low temperature + precise prompt = consistent, predictable responses
print("=== LOW TEMPERATURE (0.1) + PRECISE PROMPT ===")
for i in range(4):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="""You are a solutions architect. When asked about architecture decisions, always respond with exactly three sentences:
1. State the key trade-off in one sentence
2. Give the scenario where option A wins
3. Give the scenario where option B wins
Do not use bullet points. Write the three sentences as a single paragraph.""",
        input="Should we use microservices?",
        temperature=0.1
    )
    text = response.output_text.replace("\n", " ")[:120]
    print(f"  Run {i+1}: {text}...")

=== LOW TEMPERATURE (0.1) + PRECISE PROMPT ===


  Run 1: The key trade-off is between the flexibility and scalability of microservices versus the complexity and overhead they in...


  Run 2: The key trade-off is between the flexibility and scalability of microservices versus the complexity and overhead they in...


  Run 3: The key trade-off is between the flexibility and scalability of microservices versus the complexity of managing multiple...


  Run 4: The key trade-off with microservices is the complexity of managing multiple services versus the flexibility and scalabil...


---
# Bloque 3 — Parámetros del modelo y herramientas integradas

**Objetivo:** Conocer los parámetros esenciales que controlan el comportamiento del modelo y las herramientas integradas que la API pone a nuestra disposición.

Parámetros que veremos:
- `temperature` — controla la aleatoriedad (con nota sobre `top_p` como alternativa)
- `max_output_tokens` — límite máximo de tokens de salida
- `stop` y `seed` — control de parada y reproducibilidad

Herramientas integradas que veremos:
- `web_search` — acceso a información actual de la web
- Mención breve de otras: `file_search`, `code_interpreter`, `computer_use`

### 3.1. Temperature: de determinista a creativo

`temperature` controla cuánta aleatoriedad introduce el modelo al elegir el siguiente token. El rango típico es 0.0 – 2.0 (en Anthropic, 0.0 – 1.0).

- **0.0** — el modelo siempre elige el token más probable. Respuestas casi deterministas.
- **0.3 – 0.5** — ligera variación. Bueno para análisis técnico, clasificación, extracción de datos.
- **0.7 – 1.0** — respuestas más variadas y creativas. Bueno para brainstorming, redacción creativa.
- **> 1.0** — alta aleatoriedad. Puede producir resultados incoherentes.

**Alternativa: `top_p` (nucleus sampling).** Limita el muestreo a los tokens cuya probabilidad acumulada supere `p`. Con `top_p=0.1`, solo se consideran los tokens más probables. **Regla:** usa `temperature` O `top_p`, nunca ambos a la vez — tanto OpenAI como Anthropic lo recomiendan explícitamente, y en modelos Claude 4.5+ su uso combinado es directamente rechazado por la API.

> 🔧 **Corrección 6, nota sobre modelos recientes:** en los modelos Claude más nuevos los parámetros de muestreo han desaparecido. Con Opus 4.7+, Sonnet 5 o Fable 5, la API rechaza `temperature`, `top_p` y `top_k` con un error 400. En Haiku 5.5 y Sonnet 5.5 solo se aceptan los valores por defecto. En el resto, la versión de API `2026-10-21` también los rechaza. Lo de esta sección sigue valiendo para OpenAI y para `claude-haiku-4-5`, que es el modelo de este notebook. En los modelos nuevos, la variabilidad y la longitud se controlan con instrucciones en el prompt y con `effort`.

Veamos el efecto con una pregunta idéntica lanzada 3 veces a cada nivel:

In [15]:
TEMP_SYSTEM = "You are a product naming consultant. Propose a single product name for the given description. Respond with only the name, no explanation."
TEMP_QUESTION = "A mobile app that helps freelance developers track billable hours across multiple clients."

temperatures = [0.0, 0.7, 1.5]

for temp in temperatures:
    print(f"\n=== temperature = {temp} ===")
    for i in range(3):
        resp = openai_client.responses.create(
            model="gpt-4o-mini",
            instructions=TEMP_SYSTEM,
            input=TEMP_QUESTION,
            temperature=temp,
            max_output_tokens=30
        )
        print(f"  Run {i+1}: {resp.output_text.strip()}")


=== temperature = 0.0 ===


  Run 1: TimeTrack Pro


  Run 2: TimeTrack Pro


  Run 3: TimeTrack Pro

=== temperature = 0.7 ===


  Run 1: TimeTrack Pro


  Run 2: Billable Buddy


  Run 3: TimeTrack Pro

=== temperature = 1.5 ===


  Run 1: BillTrackr


  Run 2: TimeFolio


  Run 3: HourWise


**Observaciones esperadas:**
- Con `temperature=0.0`, las 3 respuestas son idénticas o casi idénticas.
- Con `temperature=0.7`, las respuestas son diferentes pero todas razonables.
- Con `temperature=1.5`, las respuestas pueden ser muy creativas — o incoherentes.

**Regla práctica:** empieza siempre con `temperature=0.2-0.3` para tareas de producción. Sube solo cuando necesites creatividad explícita.

### 3.2. max_output_tokens: controlando la longitud de la respuesta

`max_output_tokens` define el **techo** de tokens que el modelo puede generar. No es la longitud objetivo — es el límite superior. Si se alcanza, la respuesta se corta abruptamente.

Dos razones para usarlo en producción:
1. **Control de coste** — los tokens de salida son 3-10x más caros que los de entrada.
2. **Prevención de respuestas runaway** — evita que el modelo genere respuestas excesivamente largas.

In [16]:
LENGTH_SYSTEM = "You are a technical writer. Explain the concept clearly and thoroughly."
LENGTH_QUESTION = "Explain how database indexing works and when to use different index types."

limits = [50, 200, 800]

for limit in limits:
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=LENGTH_SYSTEM,
        input=LENGTH_QUESTION,
        temperature=0.3,
        max_output_tokens=limit
    )

    status = resp.status
    truncated = " [TRUNCATED]" if status == "incomplete" else " [COMPLETE]"

    print(f"\n=== max_output_tokens = {limit}{truncated} ===")
    print(f"Tokens generated: {resp.usage.output_tokens}")
    print(f"Response:\n{resp.output_text}")


=== max_output_tokens = 50 [TRUNCATED] ===
Tokens generated: 50
Response:
### Understanding Database Indexing

**Database Indexing** is a data structure technique that improves the speed of data retrieval operations on a database table. An index is essentially a pointer to data in a table, allowing the database management system (DBMS)



=== max_output_tokens = 200 [TRUNCATED] ===
Tokens generated: 200
Response:
### Understanding Database Indexing

Database indexing is a technique used to improve the speed of data retrieval operations on a database table. An index is a data structure that provides a quick way to look up rows in a table based on the values of one or more columns. Think of it as a book's index, which allows you to find information without reading the entire book.

#### How Indexing Works

1. **Structure of an Index**:
   - An index is typically implemented as a balanced tree (like a B-tree or B+ tree) or a hash table.
   - The index contains a sorted list of keys (the indexed column values) along with pointers to the actual rows in the table.

2. **Creating an Index**:
   - When you create an index on a column, the database engine builds the index structure and populates it with the values from that column.
   - For example, if you create an index on the "last_name" column of a "users



=== max_output_tokens = 800 [TRUNCATED] ===
Tokens generated: 800
Response:
### Understanding Database Indexing

Database indexing is a technique used to optimize the speed of data retrieval operations on a database table. An index is a data structure that improves the speed of data retrieval by providing a quick way to look up data without scanning the entire table. Think of it like an index in a book, which allows you to find information quickly without reading every page.

#### How Indexing Works

1. **Structure**: An index is typically implemented as a balanced tree (like a B-tree or B+ tree) or a hash table. The structure allows for efficient searching, inserting, and deleting of records.

2. **Creation**: When an index is created on a column (or a set of columns), the database management system (DBMS) builds the index structure based on the values in that column. For example, if you have a table of customers and you create an index on the "last_name" column, the DBMS will create

**Observaciones esperadas:**
- Con `max_output_tokens=50`, la respuesta se corta a mitad de frase. El `status` será `"incomplete"`.
- Con `max_output_tokens=800`, la respuesta termina de forma natural antes de llegar al límite.

**Regla práctica:** `max_output_tokens` es un límite de seguridad, no un control de longitud. Para controlar la longitud real, combínalo con instrucciones en el system prompt ("Maximum 150 words", "Respond in exactly 3 sentences").

⚠️ **Importante:** en Anthropic, `max_tokens` es obligatorio. En OpenAI y Gemini es opcional.

### 3.3. Herramientas integradas del modelo

Hasta ahora, nuestros LLMs solo responden con lo que saben del entrenamiento. Pero los LLMs tienen tres limitaciones fundamentales:

1. **Knowledge cutoff** — no saben nada posterior a su fecha de entrenamiento
2. **No acceden a información privada** — no pueden leer tus documentos, base de datos, etc.
3. **No ejecutan código** — no pueden calcular cosas, ejecutar SQL, procesar archivos

Las **herramientas integradas** resuelven estas limitaciones. Son funcionalidades que el proveedor ejecuta en sus propios servidores: el modelo decide cuándo usarlas, las invoca, procesa los resultados y genera la respuesta final. Todo en una sola llamada a la API desde tu lado.

**Herramientas disponibles (abril 2026):**

| Herramienta | OpenAI | Anthropic | Para qué sirve |
|-------------|:------:|:---------:|----------------|
| `web_search` | ✅ | ✅ | Buscar información actual en internet |
| `file_search` / retrieval | ✅ (con vector store) | ❌ (usa MCP) | Buscar en tus documentos |
| `code_interpreter` / `code_execution` | ✅ | ✅ | Ejecutar código Python en sandbox |
| `computer_use` | ✅ | ✅ (beta) | Controlar un ordenador (clicks, teclado) |
| `image_generation` | ✅ | ❌ | Generar imágenes |
| MCP remote servers | ✅ | ✅ | Conectar a servidores externos de herramientas |

Vamos a ver la más inmediatamente útil: `web_search`.

### 3.4. web_search con OpenAI: respuestas con información actual

La pregunta "¿qué pasó ayer en la bolsa?" es imposible de responder con el conocimiento del modelo. Con `web_search`, el modelo busca, lee páginas web relevantes, y genera una respuesta con citas.

Sintaxis en OpenAI: añades `tools=[{"type": "web_search"}]` a la llamada. El modelo decide si usar la herramienta o no.

In [17]:
# Question that requires current information
response = openai_client.responses.create(
    model="gpt-4o",
    input="What were the most important announcements at OpenAI DevDay 2025?",
    tools=[{"type": "web_search"}]
)

print("=== Response ===")
print(response.output_text)
print(f"\n=== Tokens ===")
print(f"Input: {response.usage.input_tokens} / Output: {response.usage.output_tokens}")

=== Response ===
Here’s a detailed breakdown of the most important announcements from **OpenAI DevDay 2025**, held on **October 6, 2025**, at Fort Mason in San Francisco:

---

##  Key Announcements

### 1. Apps in ChatGPT (Apps SDK)
- OpenAI unveiled **Apps in ChatGPT**, transforming ChatGPT from a chatbot into an application platform. Developers can now embed fully interactive third-party apps directly within ChatGPT conversations. These apps are built using the **Apps SDK** and the **Model Context Protocol (MCP)**, enabling seamless context sharing and fluid user experiences([infoq.com](https://www.infoq.com/news/2025/10/openai-dev-day/?utm_source=openai)).
- In demos, users shifted effortlessly from designing a poster with Canva to viewing a Zillow map, all within the same chat thread([infoq.com](https://www.infoq.com/news/2025/10/openai-dev-day/?utm_source=openai)).

### 2. AgentKit — Tools for Building Agentic Workflows
- **AgentKit** was introduced to simplify building, deployin

**Citas de fuentes:** la respuesta incluye automáticamente `annotations` que referencian las URLs consultadas. Puedes acceder a ellas iterando sobre `response.output`:

In [18]:
# Extract the URLs the model consulted
print("=== Sources consulted ===")
for item in response.output:
    if item.type == "message":
        for block in item.content:
            if hasattr(block, "annotations") and block.annotations:
                for annotation in block.annotations:
                    if hasattr(annotation, "url"):
                        print(f"  - {annotation.url}")

=== Sources consulted ===
  - https://www.infoq.com/news/2025/10/openai-dev-day/?utm_source=openai
  - https://www.infoq.com/news/2025/10/openai-dev-day/?utm_source=openai
  - https://community.openai.com/t/devday-2025-apps-sdk-sora-2-gpt-5-pro-agentkit-new-image-generation-and-speech-to-speech-mini-models-and-more/1361279?utm_source=openai
  - https://aiwiz.uk/openai-devday-2025/?utm_source=openai
  - https://openai.com/devday/2025/?utm_source=openai
  - https://openai.com/devday/2025/?utm_source=openai
  - https://openai.com/devday/2025/?utm_source=openai
  - https://openai.com/devday/2025/?utm_source=openai
  - https://openai.com/devday/2025/?utm_source=openai
  - https://openai.com/devday/2025/?utm_source=openai
  - https://www.itechguides.com/openai-devday-2025-keynote-announcements-and-innovations/?utm_source=openai


**Observaciones importantes:**
- La API gestiona todo el flujo: decide si buscar, formula la query, lee resultados, sintetiza
- El modelo cita sus fuentes automáticamente (fundamental para productos serios)
- No todos los modelos soportan `web_search`. En el ejemplo usamos `gpt-4o`. Si pruebas con `gpt-4o-mini` puede fallar.
- Tiene coste adicional sobre los tokens normales (consulta precios actualizados en la documentación)

### 3.5. web_search con Anthropic: mismo patrón, distinta sintaxis

Anthropic ofrece la misma funcionalidad con una sintaxis ligeramente distinta. La diferencia más importante: la herramienta tiene que estar **habilitada por el administrador de la organización** en la Claude Console antes de poder usarse.

In [19]:
# Note: requires web search enabled in your organization's Claude Console
# If you get an error, the feature is not enabled on your account yet.

try:
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": "What were the most important announcements at OpenAI DevDay 2025?"
        }],
        tools=[{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 3   # Cap on number of searches
        }]
    )

    # The response may contain multiple content blocks (text + tool_use + text)
    for block in response.content:
        if block.type == "text":
            print(block.text)

    # Check how many searches were performed
    if hasattr(response.usage, "server_tool_use"):
        print(f"\n[Web searches performed: {response.usage.server_tool_use.web_search_requests}]")
    print(f"[Tokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out]")

except Exception as e:
    print(f"Error: {e}")
    print("\nIf the error says web_search is not enabled, ask your org admin to enable it")
    print("in the Claude Console (Settings > Privacy > Tools).")

Based on the search results, here are the most important announcements from OpenAI DevDay 2025:

## Major Product Announcements


The event was held on October 6, 2025, at Fort Mason in San Francisco
, and 
drew more than 1,500 developers
.

**Apps in ChatGPT / Apps SDK**: 
OpenAI launched a new "apps" feature in ChatGPT that lets you bring in third-party applications from major companies like Spotify, Zillow, Canva, Booking.com, Expedia, and Figma directly inside the ChatGPT interface
. 
This is built on top of Model Context Protocols (MCPs), which OpenAI has aligned with as an open standard
.

**AgentKit**: 
OpenAI's new toolkit for building, deploying, and optimizing agents includes a visual workflow builder, an embeddable chat UI, and built-in evals (including third-party model support)
.

**New AI Models**: 
GPT-5 Pro is OpenAI's smartest and most expensive model ($15/$20) and is now generally available
. Additionally, 
Sora 2, the text-to-video model, is now available through the

**Diferencias prácticas entre ambas implementaciones:**

| Aspecto | OpenAI | Anthropic |
|---------|--------|-----------|
| Sintaxis del tool | `{"type": "web_search"}` | `{"type": "web_search_20250305", "name": "web_search"}` |
| Versionado | Implícito en el SDK | Explícito en el tool type (fecha) |
| Habilitación | Directa con API key | Requiere enable en Console |
| Control de uso | Automático | `max_uses` explícito |
| Dominios permitidos | Configurable | `allowed_domains` / `blocked_domains` |
| Pricing | Incluido en el pricing del modelo | $10 / 1.000 búsquedas + tokens |
| Citas | `annotations` en output | Citas inline en el texto |

### 3.6. Otras herramientas disponibles

No las ejecutaremos en clase, pero es importante conocerlas porque aparecerán en sesiones posteriores del programa:

**`file_search` (OpenAI) — búsqueda en tus documentos**

```python
# Requires creating a Vector Store first with your documents
response = openai_client.responses.create(
    model="gpt-4o",
    input="What is our policy on refund requests?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["<your-vector-store-id>"]
    }]
)
```

Es RAG gestionado por OpenAI: subes documentos, OpenAI los chunkea, vectoriza e indexa. El modelo busca fragmentos relevantes automáticamente. **Conectaremos con esto en el Módulo 3 cuando construyamos nuestro propio sistema RAG.**

**`code_interpreter` / `code_execution` — ejecutar código en un sandbox**

```python
# OpenAI
tools=[{"type": "code_interpreter", "container": {"type": "auto"}}]

# Anthropic
tools=[{"type": "code_execution_20250825", "name": "code_execution"}]
```

El modelo genera código Python, lo ejecuta en un entorno aislado, usa el resultado para responder. Útil para cálculos precisos, análisis de CSVs, procesamiento de archivos. **Resuelve el problema del modelo que no sabe sumar.**

**`computer_use` — controlar un ordenador**

El modelo puede ver screenshots, mover el ratón, escribir, hacer clicks. Sirve para automatizar tareas en interfaces gráficas. Todavía en beta en ambos proveedores.

**MCP (Model Context Protocol) remote servers**

Ambos proveedores soportan conectar el modelo a servidores MCP remotos. Es el estándar emergente para exponer herramientas personalizadas al modelo. **Lo veremos en el Módulo 5 cuando trabajemos con agentes.**

---

**Regla general:** las herramientas integradas transforman al LLM de "oráculo que sabe" a "agente que actúa". Son el puente entre un modelo estático y un producto realmente útil.

---
# Bloque 4 — Comparativa de proveedores en vivo

**Objetivo:** Experimentar las diferencias reales entre proveedores y entre tiers de modelos con la misma pregunta y el mismo prompt profesional.

### 4.1. Misma pregunta, distinto proveedor

Usamos un prompt de producción y una pregunta compleja para ver diferencias reales, no respuestas de juguete.

In [20]:
# Shared prompt and question for provider comparison
COMPARISON_SYSTEM_PROMPT = """You are a database architect with deep experience in both relational and NoSQL systems in production environments handling 10K+ requests per second.

When comparing technologies:
- Ground every claim in a concrete scenario (e.g., 'if your write volume exceeds X, then Y becomes a bottleneck because Z')
- Quantify trade-offs when possible (latency ranges, storage overhead percentages, operational cost implications)
- Distinguish between limitations that are fundamental vs. limitations that tooling can mitigate
- Write in prose, no bullet points, no headers
- Maximum 200 words
- Do not hedge with 'it depends' without specifying what it depends on"""

COMPARISON_QUESTION = """We have a SaaS platform with 50M rows of customer activity data growing at 2M rows/month.
Currently on PostgreSQL 16, queries are getting slow on our analytics dashboard.
Should we migrate the analytics workload to a columnar store or add read replicas and optimize indexes?"""

# 🔧 Corrección 2: aquí se redefinía PRICING; ahora usamos la tabla única de la celda de utilidades.

In [21]:
# OpenAI (Responses API)
openai_response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=COMPARISON_SYSTEM_PROMPT,
    input=COMPARISON_QUESTION,
    temperature=0.2,
    max_output_tokens=400
)

openai_cost = calculate_cost(
    openai_response.model,
    openai_response.usage.input_tokens,
    openai_response.usage.output_tokens
)

print("=== OpenAI (gpt-4o-mini) ===")
print(openai_response.output_text)
print(f"\nModel: {openai_response.model}")
print(f"Tokens: {openai_response.usage.input_tokens} input + {openai_response.usage.output_tokens} output")
if openai_cost:
    print(f"Cost: ${openai_cost['total']:.6f}")

=== OpenAI (gpt-4o-mini) ===
Given your current workload and growth rate, migrating the analytics workload to a columnar store like Amazon Redshift or Google BigQuery could significantly enhance query performance. Columnar databases excel in read-heavy scenarios, particularly for analytical queries, as they store data in a way that optimizes aggregation and filtering. For instance, if your queries involve scanning large datasets for analytics, a columnar store can reduce I/O by up to 90%, translating to latency improvements from seconds to milliseconds for complex aggregations.

On the other hand, adding read replicas and optimizing indexes in PostgreSQL can alleviate some immediate performance issues. However, this approach may only provide marginal gains, especially as your dataset grows. Indexing can improve query performance by 20-50%, but with 50M rows and continuous growth, the overhead of maintaining these indexes can lead to increased write latency and storage overhead of 10-30

In [22]:
# Anthropic (Messages API)
anthropic_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=COMPARISON_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": COMPARISON_QUESTION}],
    max_tokens=400
)

anthropic_cost = calculate_cost(
    anthropic_response.model,
    anthropic_response.usage.input_tokens,
    anthropic_response.usage.output_tokens
)

print("=== Anthropic (claude-haiku-4-5) ===")
print(anthropic_text(anthropic_response))  # 🔧 Corrección 4
warn_if_truncated(anthropic_response)  # 🔧 Corrección 5
print(f"\nModel: {anthropic_response.model}")
print(f"Tokens: {anthropic_response.usage.input_tokens} input + {anthropic_response.usage.output_tokens} output")
if anthropic_cost:
    print(f"Cost: ${anthropic_cost['total']:.6f}")

=== Anthropic (claude-haiku-4-5) ===
Read replicas won't solve your actual problem. At 2M rows monthly, you're accumulating 300M rows annually—your analytics queries are slow because they're scanning wide tables sequentially, not because write contention is saturating your primary. Adding replicas just duplicates the inefficiency.

A columnar store (Clickhouse, DuckDB, or Redshift) is the right lever here. Your analytics dashboard likely aggregates activity across date ranges and customer segments—columnar compression will give you 5-10x storage reduction and 50-100x query speedup on those access patterns because you're only materializing relevant columns, not entire rows.

The migration path depends on your analytics freshness requirements. If you need sub-minute currency, set up a streaming pipeline (Kafka/Fivetran → Clickhouse) alongside your PostgreSQL primary. If hourly snapshots suffice, a nightly batch extract costs virtually nothing and requires zero pipeline complexity.

The r

In [23]:
# Side-by-side comparison
print("=" * 70)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(f"{'':30s} {'OpenAI':>15s}  {'Anthropic':>15s}")
print("-" * 70)
print(f"{'Input tokens':30s} {openai_response.usage.input_tokens:>15d}  {anthropic_response.usage.input_tokens:>15d}")
print(f"{'Output tokens':30s} {openai_response.usage.output_tokens:>15d}  {anthropic_response.usage.output_tokens:>15d}")
if openai_cost and anthropic_cost:
    # 🔧 Corrección 8: el original anidaba comillas dobles dentro del f-string, algo válido
    # solo desde Python 3.12 (PEP 701). Colab lo acepta, pero Python 3.11 da SyntaxError.
    openai_total = f"${openai_cost['total']:.6f}"
    anthropic_total = f"${anthropic_cost['total']:.6f}"
    print(f"{'Total cost':30s} {openai_total:>15s}  {anthropic_total:>15s}")

SIDE-BY-SIDE COMPARISON
                                        OpenAI        Anthropic
----------------------------------------------------------------------
Input tokens                               199              218
Output tokens                              213              288
Total cost                           $0.000158        $0.001658


### 4.2. Misma pregunta, distinto tier de modelo

Comparamos modelos de distintos precios dentro de OpenAI con la misma pregunta compleja. ¿Justifica la diferencia de precio la diferencia de calidad?

In [24]:
# Compare different OpenAI model tiers with a complex question
openai_models = [
    "gpt-4o-mini",
    "gpt-5.4-mini",   # 🔧 Corrección 7: ya estaban activos; comenta los que no tengas disponibles
    "gpt-5.4",        # (more expensive)
]

for model_name in openai_models:
    try:
        resp = openai_client.responses.create(
            model=model_name,
            instructions=COMPARISON_SYSTEM_PROMPT,
            input=COMPARISON_QUESTION,
            temperature=0.2,
            max_output_tokens=400
        )

        cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)

        print(f"\n{'=' * 60}")
        print(f"Model: {resp.model}")
        print(f"Tokens: {resp.usage.input_tokens} input + {resp.usage.output_tokens} output")
        if cost:
            print(f"Cost: ${cost['total']:.6f}")
        print(f"\nResponse:\n{resp.output_text}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"Model {model_name}: Error — {e}")


Model: gpt-4o-mini-2024-07-18
Tokens: 199 input + 218 output
Cost: $0.000161

Response:
Given your current scale of 50 million rows and a growth rate of 2 million rows per month, the performance degradation in your PostgreSQL analytics dashboard likely stems from the increasing complexity of queries and the volume of data being processed. If your read volume is high and queries are primarily analytical, migrating to a columnar store like Amazon Redshift or Google BigQuery could significantly enhance performance. Columnar databases excel in read-heavy workloads, often reducing query times from seconds to milliseconds, especially for aggregations and scans over large datasets.

On the other hand, adding read replicas and optimizing indexes can provide immediate relief but may not scale effectively as your data grows. While read replicas can distribute the load, they introduce replication lag, which can affect real-time analytics. Index optimization can improve query performance but may 


Model: gpt-5.4-mini-2026-03-17
Tokens: 198 input + 266 output
Cost: $0.001346

Response:
If your dashboard is mostly aggregations over large scans, a columnar store will usually beat PostgreSQL replicas by a wide margin once queries touch more than about 1–5% of the 50M-row table. In that case, even a well-indexed Postgres replica still has to read many heap pages and often lands in the 100s of milliseconds to seconds for group-bys over months of data, while a columnar engine can cut that to tens of milliseconds to low hundreds because it reads only the needed columns and compresses them heavily, often 3–10x smaller on disk.

If, instead, the dashboard is slow because of a few selective filters, missing composite indexes, or bad plans, then read replicas plus index tuning is the cheaper fix. A replica can absorb read load immediately, but it does not fundamentally change the cost of wide scans; it just moves them off the primary. If your slow queries are returning under ~0.1% of rows 


Model: gpt-5.4-2026-03-05
Tokens: 198 input + 303 output
Cost: $0.005040

Response:
At 50M rows growing by 2M/month, PostgreSQL is not near a fundamental size limit; slow dashboards usually mean row-store mismatch for scan-heavy aggregates, missing partitioning, or indexes that help point lookups but not “GROUP BY over 6 months.” Read replicas will not make a 4-second query become 200 ms; they add read capacity, not per-query speed, unless your primary is CPU or I/O saturated from concurrent dashboard traffic. In a concrete case, if 20 users each run queries scanning 10–20M rows, a replica can cut contention and keep OLTP healthy, but each query may still take 2–8 seconds.

If your dashboard mostly does filtered aggregates over many rows, a columnar store is the structural fix. Columnar engines often cut scanned bytes by 5–20x and make those queries 3–15x faster, often turning multi-second scans into sub-second or low-single-second results. The cost is ETL/CDC complexity, duplicate st

### 🧪 Tu turno

Haz la misma comparación con modelos de Anthropic. Prueba con `claude-haiku-4-5` y si tienes acceso, con `claude-sonnet-4-6`.

In [25]:
# Compare different Anthropic model tiers
anthropic_models = [
    "claude-haiku-4-5",
    "claude-sonnet-4-6",  # 🔧 Corrección 7: comenta si no tienes acceso
]

for model_name in anthropic_models:
    try:
        resp = anthropic_client.messages.create(
            model=model_name,
            system=COMPARISON_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": COMPARISON_QUESTION}],
            max_tokens=400
        )

        cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)

        print(f"\n{'=' * 60}")
        print(f"Model: {resp.model}")
        print(f"Tokens: {resp.usage.input_tokens} input + {resp.usage.output_tokens} output")
        if cost:
            print(f"Cost: ${cost['total']:.6f}")
        print(f"\nResponse:\n{anthropic_text(resp)}")  # 🔧 Corrección 4
        warn_if_truncated(resp)  # 🔧 Corrección 5
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"Model {model_name}: Error — {e}")


Model: claude-haiku-4-5-20251001
Tokens: 218 input + 320 output
Cost: $0.001818

Response:
At 2M rows/month, you're hitting the wall where row-oriented storage becomes inefficient for analytics—likely experiencing 10-100x slower scans than necessary because PostgreSQL reads entire rows when you only need 2-3 columns per query.

Read replicas won't solve this. They distribute load but don't change the fundamental I/O pattern. Your analytics queries still scan full rows; you're just doing it on a different machine. You'd need aggressive indexing, which creates write amplification that hurts your transactional workload at 2M inserts/month.

Columnar is the right architectural choice here. Switching to Redshift, BigQuery, or ClickHouse will give you 20-50x faster analytics queries because they only touch the columns you need. Storage overhead drops 3-5x compared to row-oriented with compression.

The catch: columnar stores are poor at point lookups and real-time transactional updates. If 


Model: claude-sonnet-4-6
Tokens: 219 input + 335 output
Cost: $0.005682

Response:
At 50M rows growing 2M/month, you're not yet at a scale where PostgreSQL fundamentally breaks—you're at the scale where *unoptimized* PostgreSQL breaks. The decision hinges on your query patterns.

If your dashboard runs aggregations across many rows but few columns (e.g., `SUM(revenue) GROUP BY customer_id` scanning 40M rows), that's a columnar access pattern. PostgreSQL's row storage reads entire 8KB pages even when you need 2 of 15 columns, burning 3-5x unnecessary I/O. ClickHouse or Redshift would cut those queries from 8-15 seconds to under 500ms because columnar compression and vectorized execution are purpose-built for this. The operational cost is a real-time sync pipeline (Debezium + Kafka, roughly 200-400ms lag), which matters if dashboards need live data.

If your slowness comes from missing partial indexes, unvacuumed bloat, or JOIN-heavy reports on indexed foreign keys, read replicas with `

---
# Bloque 5 — Agregadores: una interfaz, múltiples proveedores

**Objetivo:** Cambiar de proveedor y modelo con una sola línea de código usando LiteLLM.

In [26]:
from litellm import completion

AGGREGATOR_SYSTEM_PROMPT = """You are a backend engineering lead evaluating infrastructure decisions.
Respond in exactly 3 sentences: the recommendation, the primary reason, and one risk to watch for.
Be specific — reference concrete thresholds, metrics, or failure modes. No generic advice."""

AGGREGATOR_QUESTION = "We process 500 webhook events per second from Stripe. Should we handle them synchronously in our Rails API or push them to a message queue first?"

# Same function, same interface — only the model string changes
providers = [
    "openai/gpt-4o-mini",
    "anthropic/claude-haiku-4-5",
]

for model_name in providers:
    try:
        resp = completion(
            model=model_name,
            messages=[
                {"role": "system", "content": AGGREGATOR_SYSTEM_PROMPT},
                {"role": "user", "content": AGGREGATOR_QUESTION}
            ],
            temperature=0.2,
            max_tokens=250
        )

        print(f"\n{'=' * 60}")
        print(f"Model: {model_name}")
        print(f"Tokens: {resp.usage.prompt_tokens} input + {resp.usage.completion_tokens} output")
        print(f"\nResponse:\n{resp.choices[0].message.content}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"{model_name}: Error — {e}")


Model: openai/gpt-4o-mini
Tokens: 89 input + 110 output

Response:
I recommend pushing the webhook events to a message queue first, such as RabbitMQ or AWS SQS, to decouple processing from the API response. This approach allows you to handle bursts of up to 1,000 events per second without overwhelming your Rails application, which could lead to increased latency or timeouts if processed synchronously. One risk to watch for is message backlog; if processing fails or slows down, ensure you have monitoring in place for queue length metrics, ideally keeping it below 1,000 messages to avoid delays in event handling.



Model: anthropic/claude-haiku-4-5
Tokens: 94 input + 224 output

Response:
**Recommendation:** Push to a message queue (SQS/RabbitMQ) immediately and process asynchronously, keeping synchronous handling only for Stripe signature validation and acknowledgment.

**Primary reason:** At 500 events/sec, synchronous processing will exhaust Rails connection pools (default ~5 connections) within milliseconds if any downstream dependency (database writes, API calls) takes >10ms; you'll hit timeout cascades and lose webhook deliveries. Stripe expects a 2xx response within 5 seconds—a queue decouples that constraint and lets you process at your actual throughput capacity (measure current p99 latency per event; if >50ms, async is mandatory).

**Risk to watch:** Message queue backlog growth during traffic spikes or downstream outages—set up CloudWatch alarms for queue depth >10k messages and implement dead-letter queue routing for events that fail after 3 retries, or you'll silently lose reconcili

### Observación importante

LiteLLM usa la interfaz de **Chat Completions** como formato unificado:
- `messages` en lugar de `instructions` + `input`
- `resp.choices[0].message.content` en lugar de `resp.output_text`
- `resp.usage.prompt_tokens` en lugar de `resp.usage.input_tokens`

Por eso es importante conocer ambas interfaces de OpenAI: la **Responses API** es la que usamos directamente, y **Chat Completions** es el "lenguaje común" de los agregadores.

### Wrapper con cambio de proveedor en una línea

Así se ve una función de producción que abstrae completamente el proveedor:

In [27]:
def evaluate_technical_decision(
    scenario: str,
    model: str = "openai/gpt-4o-mini"
) -> str:
    """
    Evaluate a technical decision given a scenario description.
    Switching providers = changing the model string. No other code changes.
    """
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": """You are a principal engineer performing a technical design review.
For every decision, provide: the recommended approach, one concrete reason grounded in production experience, and one thing that could go wrong.
Maximum 3 sentences. No bullet points."""},
            {"role": "user", "content": scenario}
        ],
        temperature=0.2,
        max_tokens=200
    )
    return resp.choices[0].message.content


scenario = "We want to store user session data. Considering Redis vs PostgreSQL for a system with 20K concurrent users."

# Switching providers = changing one string
print("OpenAI:")
print(evaluate_technical_decision(scenario, "openai/gpt-4o-mini"))
print()
print("Anthropic:")
print(evaluate_technical_decision(scenario, "anthropic/claude-haiku-4-5-20251001"))

OpenAI:


I recommend using Redis for storing user session data due to its in-memory data structure store, which provides extremely low latency and high throughput, essential for handling 20K concurrent users effectively. In my production experience, Redis has proven to significantly reduce response times for session retrieval, enhancing user experience during peak loads. However, one potential issue could be data persistence; if not configured properly, you risk losing session data during a Redis server restart or failure.

Anthropic:


**Recommendation: Redis as primary with PostgreSQL for persistence.**

Redis handles 20K concurrent sessions with sub-millisecond latency and built-in expiration, whereas PostgreSQL would require polling or application logic to manage session lifecycle—I've seen PostgreSQL session stores become bottlenecks under concurrent load when queries compete with other workloads. The risk is data loss during Redis restarts if you don't configure AOF/RDB persistence properly, so implement Redis Sentinel or Cluster with synchronous replication to a secondary instance.


---
# Bloque 6 — Tokens: cómo funcionan y cuánto cuestan

**Objetivo:** Entender los tokens de forma práctica — qué son, cómo se cuentan, y cómo impactan en coste.

### 6.1. Anatomía de los tokens

In [28]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

sample_texts = {
    "Short English": "How much does it cost to migrate a database?",
    "Short Spanish": "¿Cuánto cuesta migrar una base de datos?",
    "Python code": "def migrate_db(source, target):\n    return source.copy_to(target)",
    "JSON payload": '{"project": "migration", "estimate_days": 45, "confidence": "medium"}',
    "Production prompt": PRODUCTION_SYSTEM_PROMPT,  # The prompt we built in Block 2
}

print(f"{'Text':25s} {'Characters':>12s} {'Tokens':>8s} {'Tok/word':>10s}")
print("-" * 59)

for label, text in sample_texts.items():
    tokens = enc.encode(text)
    word_count = len(text.split())
    ratio = len(tokens) / word_count if word_count > 0 else 0
    print(f"{label:25s} {len(text):>12d} {len(tokens):>8d} {ratio:>9.1f}x")

Text                        Characters   Tokens   Tok/word
-----------------------------------------------------------
Short English                       44       10       1.1x
Short Spanish                       40       10       1.4x
Python code                         65       14       2.8x
JSON payload                        69       19       3.2x
Production prompt                 1560      295       1.3x


In [29]:
# Visualize how a text is decomposed into tokens
sample = "¿Cuánto cuesta migrar una base de datos PostgreSQL a Amazon Aurora?"
tokens = enc.encode(sample)

print(f"Text: {sample}")
print(f"Total: {len(tokens)} tokens\n")
print("Decomposition:")
for i, t in enumerate(tokens):
    print(f"  Token {i}: '{enc.decode([t])}' (id: {t})")

Text: ¿Cuánto cuesta migrar una base de datos PostgreSQL a Amazon Aurora?
Total: 15 tokens

Decomposition:
  Token 0: '¿Cu' (id: 102091)
  Token 1: 'ánto' (id: 110808)
  Token 2: ' cuesta' (id: 116278)
  Token 3: ' migr' (id: 16848)
  Token 4: 'ar' (id: 277)
  Token 5: ' una' (id: 1969)
  Token 6: ' base' (id: 3611)
  Token 7: ' de' (id: 334)
  Token 8: ' datos' (id: 18336)
  Token 9: ' Postgre' (id: 145223)
  Token 10: 'SQL' (id: 10430)
  Token 11: ' a' (id: 261)
  Token 12: ' Amazon' (id: 9529)
  Token 13: ' Aurora' (id: 72767)
  Token 14: '?' (id: 30)


### 🧪 Tu turno

Tokeniza textos en tu idioma, con jerga técnica de tu dominio, o el system prompt de la función que creaste en el bloque 2. ¿Cuántos tokens consume?

In [30]:
# Your text here
my_text = "..."

tokens = enc.encode(my_text)
print(f"'{my_text}' -> {len(tokens)} tokens")

'...' -> 1 tokens


### 6.2. Input vs output: la asimetría que define tu factura

In [31]:
# Experiment: same question, different output limits
TOKEN_SYSTEM_PROMPT = "You are a technical assistant. Respond in Spanish."
TOKEN_QUESTION = "Explain the CAP theorem and when each trade-off matters."

output_limits = [50, 200, 1000]

print(f"{'max_output_tokens':>20s} {'Input tok':>12s} {'Output tok':>12s} {'Cost':>12s}")
print("-" * 60)

for limit in output_limits:
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=TOKEN_SYSTEM_PROMPT,
        input=TOKEN_QUESTION,
        max_output_tokens=limit,
        temperature=0.3
    )

    cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)
    cost_str = f"${cost['total']:.6f}" if cost else "N/A"

    print(f"{limit:>20d} {resp.usage.input_tokens:>12d} {resp.usage.output_tokens:>12d} {cost_str:>12s}")

   max_output_tokens    Input tok   Output tok         Cost
------------------------------------------------------------


                  50           32           50    $0.000035


                 200           32          200    $0.000125


                1000           32          537    $0.000327


In [32]:
# Scale projection: how much does this cost in production?

USERS = 10_000
CALLS_PER_USER_PER_DAY = 5
CALLS_PER_DAY = USERS * CALLS_PER_USER_PER_DAY
DAYS_PER_MONTH = 30
CALLS_PER_MONTH = CALLS_PER_DAY * DAYS_PER_MONTH

# Average call: 150 input tokens (production prompt is longer), 200 output tokens
AVG_INPUT_TOKENS = 150
AVG_OUTPUT_TOKENS = 200

models_to_compare = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "claude-haiku-4-5": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
}

print(f"Scenario: {USERS:,} users x {CALLS_PER_USER_PER_DAY} calls/day x {DAYS_PER_MONTH} days")
print(f"Total: {CALLS_PER_MONTH:,} calls/month")
print(f"Average per call: {AVG_INPUT_TOKENS} input tokens + {AVG_OUTPUT_TOKENS} output tokens")
print(f"\n{'Model':25s} {'Cost/call':>15s} {'Cost/month':>15s}")
print("-" * 58)

for model_name, prices in models_to_compare.items():
    cost_per_call = (
        (AVG_INPUT_TOKENS / 1_000_000) * prices["input"] +
        (AVG_OUTPUT_TOKENS / 1_000_000) * prices["output"]
    )
    monthly_cost = cost_per_call * CALLS_PER_MONTH
    print(f"{model_name:25s} ${cost_per_call:>14.6f} ${monthly_cost:>13.2f}")

Scenario: 10,000 users x 5 calls/day x 30 days
Total: 1,500,000 calls/month
Average per call: 150 input tokens + 200 output tokens

Model                           Cost/call      Cost/month
----------------------------------------------------------
gpt-4o-mini               $      0.000142 $       213.75
gpt-5.4-mini              $      0.001013 $      1518.75
gpt-5.4                   $      0.003375 $      5062.50
claude-haiku-4-5          $      0.001150 $      1725.00
claude-sonnet-4-6         $      0.003450 $      5175.00


### 6.3. El coste oculto de las conversaciones multi-turno

In [33]:
# Multi-turn conversation: watch input tokens grow with each turn

conversation_history = []
conversation_system_prompt = """You are a DevOps engineer helping a team debug a production incident.
Ask clarifying questions when needed. Keep each response under 3 sentences.
Focus on actionable next steps, not theory."""

questions = [
    "Our API response times spiked from 200ms to 3s about 30 minutes ago.",
    "PostgreSQL CPU is at 95%. No recent deployments.",
    "We found a query doing a sequential scan on a 40M row table. It was fine yesterday.",
    "The table was auto-vacuumed last night. Could that have invalidated the index statistics?",
    "Running ANALYZE on the table now. What else should we check while we wait?",
]

print(f"{'Turn':>6s} {'Input tok':>12s} {'Output tok':>12s} {'Cumulative cost':>18s}")
print("-" * 52)

cumulative_cost = 0

for i, question in enumerate(questions):
    conversation_history.append({"role": "user", "content": question})

    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=conversation_system_prompt,
        input=conversation_history,
        max_output_tokens=200,
        temperature=0.2
    )

    conversation_history.append({"role": "assistant", "content": resp.output_text})

    cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)
    if cost:
        cumulative_cost += cost["total"]

    print(f"{i+1:>6d} {resp.usage.input_tokens:>12d} {resp.usage.output_tokens:>12d} ${cumulative_cost:>16.6f}")

  Turn    Input tok   Output tok    Cumulative cost
----------------------------------------------------


     1           69           37 $        0.000033


     2          126           41 $        0.000076


     3          195           50 $        0.000135


     4          272           70 $        0.000218


     5          367           50 $        0.000303


In [34]:
# Visualize the cost of the system prompt across turns
system_prompt_tokens = len(enc.encode(conversation_system_prompt))

print(f"\n📈 Input token growth per turn:")
print(f"Each turn resends the entire conversation history as input tokens.")
print(f"\nSystem prompt in this example: ~{system_prompt_tokens} tokens")
print(f"The production prompt from Block 2: ~{len(enc.encode(PRODUCTION_SYSTEM_PROMPT))} tokens")
print(f"\nOver a 20-turn conversation with the production prompt:")
print(f"  System prompt alone = {len(enc.encode(PRODUCTION_SYSTEM_PROMPT))} x 20 = {len(enc.encode(PRODUCTION_SYSTEM_PROMPT)) * 20:,} input tokens")


📈 Input token growth per turn:
Each turn resends the entire conversation history as input tokens.

System prompt in this example: ~38 tokens
The production prompt from Block 2: ~295 tokens

Over a 20-turn conversation with the production prompt:
  System prompt alone = 295 x 20 = 5,900 input tokens


---
# Bloque 7 — Análisis de arquitecturas reales (discusión)

Este bloque es de discusión guiada por el instructor — no hay código que ejecutar.

**Preguntas para analizar cada producto:**

1. ¿Cómo abstrae el prompting del usuario?
2. ¿Qué modelo probablemente usa?
3. ¿Cómo gestiona los tokens?
4. ¿Usa model routing (modelos diferentes para tareas diferentes)?

**Productos sugeridos:**
- GitHub Copilot
- Asistente de soporte (Intercom / Zendesk AI)
- Herramienta de análisis de documentos (NotebookLM)

---
# Extra: Resumen de las tres interfaces API

Referencia rápida de las tres interfaces que hemos usado hoy.

In [35]:
# The three API interfaces used today, side by side:

SUMMARY_SYSTEM = """You are a senior engineer. When asked a technical question, respond with exactly one sentence that a principal engineer would find useful. No filler."""
SUMMARY_QUESTION = "What is the most common mistake teams make when adopting Kubernetes?"

# 1. OpenAI Responses API (recommended for OpenAI direct usage)
resp1 = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SUMMARY_SYSTEM,        # system prompt as top-level parameter
    input=SUMMARY_QUESTION,              # user input as string
    max_output_tokens=100,
    temperature=0.1
)
print("1. Responses API:", resp1.output_text)
print(f"   Tokens: {resp1.usage.input_tokens} in / {resp1.usage.output_tokens} out")
print(f"   Status: {resp1.status}")

print()

# 2. Anthropic Messages API
resp2 = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=SUMMARY_SYSTEM,               # system prompt as separate parameter
    messages=[{"role": "user", "content": SUMMARY_QUESTION}],  # messages array
    max_tokens=100
)
print("2. Messages API:", anthropic_text(resp2))  # 🔧 Corrección 4
print(f"   Tokens: {resp2.usage.input_tokens} in / {resp2.usage.output_tokens} out")
print(f"   Stop reason: {resp2.stop_reason}")

print()

# 3. LiteLLM (unified Chat Completions interface)
resp3 = completion(
    model="openai/gpt-4o-mini",
    messages=[
        {"role": "system", "content": SUMMARY_SYSTEM},   # system as message in array
        {"role": "user", "content": SUMMARY_QUESTION}
    ],
    max_tokens=100,
    temperature=0.1
)
print("3. LiteLLM:", resp3.choices[0].message.content)
print(f"   Tokens: {resp3.usage.prompt_tokens} in / {resp3.usage.completion_tokens} out")
print(f"   Finish reason: {resp3.choices[0].finish_reason}")

1. Responses API: The most common mistake teams make when adopting Kubernetes is underestimating the complexity of managing stateful applications and persistent storage.
   Tokens: 51 in / 25 out
   Status: completed



2. Messages API: The most common mistake is attempting to lift-and-shift monolithic applications directly into Kubernetes without first refactoring for statelessness, proper resource requests/limits, and observability—treating it as a container orchestrator rather than a platform that demands architectural changes.
   Tokens: 50 in / 58 out
   Stop reason: end_turn



3. LiteLLM: The most common mistake teams make when adopting Kubernetes is underestimating the complexity of managing its ecosystem and not investing enough in proper training and operational practices.
   Tokens: 51 in / 30 out
   Finish reason: stop
